# Qwen-14B Inference on TPU v2-8 (Free Colab)

This notebook loads and runs Qwen-14B (~28 GB in bfloat16) on a Colab TPU v2-8
(8 cores × 8 GB = 64 GB HBM total) by sharding the model across all 8 cores via
PyTorch/XLA SPMD. Each of the 4 cells below solves a specific hardware trap.

In [ ]:
# ─── Cell 1: TPU Initialisation & Sanity Check ────────────────────────────────
# TRAP: importing torch_xla before setting PJRT_DEVICE causes it to fall back to
# a CPU runtime; every subsequent .to('xla:0') call silently goes to CPU.

import os

# Must be set BEFORE any torch_xla import.
os.environ['PJRT_DEVICE'] = 'TPU'

# XLA_USE_BF16=1 forces all float32 ops to bfloat16 inside XLA without requiring
# explicit model-side casting — saves scratchpad memory during compilation.
os.environ['XLA_USE_BF16'] = '1'

import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr

# ── Verify TPU connectivity ────────────────────────────────────────────────────
try:
    # xr.global_device_count() queries the PJRT backend; raises if TPU is absent.
    num_devices = xr.global_device_count()
    assert num_devices == 8, (
        f"Expected 8 TPU cores, got {num_devices}. "
        "Make sure the Colab runtime is set to TPU v2-8."
    )
    print(f"✓ TPU connected — {num_devices} XLA devices available (v2-8).")
    print(f"  Local devices : {xm.get_xla_supported_devices()}")
    print(f"  Global devices: {xr.global_device_count()} cores across the pod")
except Exception as exc:
    raise RuntimeError(
        "CRITICAL: No TPU detected. Switch the Colab runtime to TPU v2-8 "
        "via Runtime → Change runtime type."
    ) from exc

In [ ]:
# ─── Cell 2: Lazy Model Loading — No OOM ──────────────────────────────────────
# TRAP 1 (8-core OOM): loading with .from_pretrained(...) and then .to('xla:0')
#   stuffs 28 GB into one 8-GB core → immediate HBM OOM.
# TRAP 2 (CPU RAM spike): keeping full weights in CPU RAM while sharding to HBM
#   doubles peak RAM usage and crashes the host VM.
#
# FIX: load on the 'meta' device (zero bytes allocated anywhere), then use
# Hugging Face `accelerate`'s init_empty_weights + load_checkpoint_and_dispatch
# to stream weights shard-by-shard directly into the XLA SPMD mesh, bypassing
# both the CPU RAM spike and the single-core HBM bottleneck.

# Install runtime dependencies — run this once per Colab session.
# torch-xla is pre-installed on Colab TPU runtimes; the others are not.
!pip install -q transformers accelerate sentencepiece

import torch
import torch_xla.core.xla_model as xm
import torch_xla.distributed.spmd as xs
import torch_xla.runtime as xr
from torch_xla.distributed.spmd import Mesh

from transformers import AutoTokenizer, AutoModelForCausalLM
from accelerate import init_empty_weights, load_checkpoint_and_dispatch

MODEL_ID = 'Qwen/Qwen-14B'

# ── 2-a  Build the SPMD mesh BEFORE any weight materialisation ────────────────
# A 2-D mesh (data=1, model=8) gives pure tensor-parallelism across all 8 cores.
# No data-parallel axis is needed for single-batch inference.
num_devices = xr.global_device_count()          # 8
mesh_shape  = (1, num_devices)                  # (data_parallel=1, model_parallel=8)
device_ids  = list(range(num_devices))
mesh = Mesh(
    device_ids=device_ids,
    mesh_shape=mesh_shape,
    axis_names=('data', 'model'),
)
xs.set_global_mesh(mesh)
print(f"SPMD mesh: {mesh}")

# ── 2-b  Tokenizer (small, safe to load normally) ─────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
print("Tokenizer loaded.")

# ── 2-c  Shell model on 'meta' — zero bytes allocated ─────────────────────────
# init_empty_weights replaces every nn.Parameter with a meta-tensor:
# the model graph is built but NO data lives in RAM or HBM.
with init_empty_weights():
    model_shell = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
print("Shell model (meta tensors) built. 0 bytes allocated.")

# ── 2-d  Stream checkpoints → XLA devices via accelerate dispatch ─────────────
# load_checkpoint_and_dispatch reads each tensor shard from disk, sends it
# to the designated device, and discards the CPU copy immediately.
# The 3.5 GiB ceiling per core leaves ~4.5 GiB of XLA compiler scratchpad.
MAX_MEMORY_PER_CORE = '3.5GiB'
device_map = {
    f'xla:{i}': MAX_MEMORY_PER_CORE for i in range(num_devices)
}

model = load_checkpoint_and_dispatch(
    model_shell,
    MODEL_ID,                    # can be a local path or HF hub id
    device_map=device_map,
    dtype=torch.bfloat16,
    no_split_module_classes=['QWenBlock'],   # never split a transformer block
)
model.eval()
print("Model loaded and dispatched to TPU cores.")

In [ ]:
# ─── Cell 3: Sharding & Distribution Verification ─────────────────────────────
# Prove that the 28 GB model is NOT bottlenecked on a single core.

from collections import defaultdict
import torch_xla.core.xla_model as xm

def bytes_to_gib(n: int) -> float:
    return n / (1024 ** 3)

# ── 3-a  Report which device each weight matrix lives on ──────────────────────
print("=" * 60)
print("Sub-module → XLA device mapping")
print("=" * 60)
for name, param in model.named_parameters():
    dev = str(param.device)
    if 'weight' in name and param.ndim >= 2:   # show only weight matrices
        print(f"  {name:<55s} {dev}")

# ── 3-b  Per-core parameter bytes ─────────────────────────────────────────────
# Tally bytes assigned to each 'xla:N' device.
core_bytes: dict[str, int] = defaultdict(int)
total_bytes = 0
for param in model.parameters():
    nbytes = param.numel() * param.element_size()
    core_bytes[str(param.device)] += nbytes
    total_bytes += nbytes

print("\n" + "=" * 60)
print("HBM usage per TPU core (parameters only)")
print("=" * 60)
for dev in sorted(core_bytes):
    gib = bytes_to_gib(core_bytes[dev])
    bar = '█' * int(gib * 4)       # 1 block ≈ 0.25 GiB
    print(f"  {dev:<10s}: {gib:5.2f} GiB  {bar}")
print(f"  {'TOTAL':<10s}: {bytes_to_gib(total_bytes):5.2f} GiB")

# ── 3-c  Verify no core is carrying more than 150% of ideal share ─────────────
ideal_gib = bytes_to_gib(total_bytes) / len(core_bytes)
for dev, nb in core_bytes.items():
    gib = bytes_to_gib(nb)
    assert gib < ideal_gib * 1.5, (
        f"SHARD IMBALANCE on {dev}: {gib:.2f} GiB "
        f"(ideal {ideal_gib:.2f} GiB, threshold {ideal_gib*1.5:.2f} GiB). "
        "Check no_split_module_classes."
    )
print("\n✓ Sharding balance check passed — model is distributed across all cores.")

# ── 3-d  Force XLA graph compilation now (warm-up) ────────────────────────────
# mark_step() flushes pending XLA IR to the compiler so first-token latency
# does not include compilation time during actual inference.
xm.mark_step()
print("✓ XLA warm-up step executed.")

In [ ]:
# ─── Cell 4: Synchronized Inference ───────────────────────────────────────────
# TRAP: calling model.generate() without synchronisation leaves XLA IR queued on
# each core independently; the host blocks indefinitely waiting for output tokens
# that the compiler never finalises → the notebook hangs.
#
# FIX: wrap every tensor move to XLA with xm.mark_step() so the compiler emits a
# complete executable, then call xm.mark_step() again after generation to flush
# the output graph before decoding on CPU.

import torch
import torch_xla.core.xla_model as xm

PROMPT = "Explain the architecture of a neural network."
MAX_NEW_TOKENS = 256

def run_inference(prompt: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """
    Run sharded inference on the Qwen-14B model across all 8 TPU cores.

    Steps
    -----
    1. Tokenise on CPU.
    2. Move input tensors to the primary XLA device (xla:0).  Because the model
       is sharded via accelerate's device_map, every inter-layer transfer is
       handled by XLA's collective-communication primitives automatically — no
       manual all-gather needed.
    3. Flush the input-copy graph with mark_step() so the compiler can overlap
       the copy with the first attention layer.
    4. Call model.generate() with standard HF args; XLA's lazy execution builds
       the decoding graph across all 8 cores.
    5. Flush the decoding graph with a second mark_step() before moving logits
       to CPU for decode — prevents the host from blocking on uncommitted IR.
    """
    # ── Step 1: tokenise ──────────────────────────────────────────────────────
    inputs = tokenizer(prompt, return_tensors='pt')

    # ── Step 2: move to primary XLA device ───────────────────────────────────
    primary_device = xm.xla_device()           # 'xla:0'
    input_ids      = inputs['input_ids'].to(primary_device)
    attention_mask = inputs['attention_mask'].to(primary_device)

    # ── Step 3: flush input-copy graph ───────────────────────────────────────
    xm.mark_step()

    # ── Step 4: generate ─────────────────────────────────────────────────────
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # greedy decoding — deterministic, faster compilation
            pad_token_id=tokenizer.eos_token_id,
        )

    # ── Step 5: flush decoding graph before host-side decode ─────────────────
    # This mark_step() is the critical synchronisation barrier: it forces XLA
    # to materialise the output tensor on device before we call .cpu(), which
    # would otherwise trigger an implicit (and potentially race-y) flush.
    xm.mark_step()

    # ── Decode on CPU ─────────────────────────────────────────────────────────
    # Slice away the prompt tokens so we print only the generated response.
    generated_ids = output_ids[0, input_ids.shape[-1]:].cpu()
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


# ── Run inference and print result ────────────────────────────────────────────
print(f"Prompt : {PROMPT}\n")
response = run_inference(PROMPT)
print("Response:")
print(response)